# 08 – Evaluation: Métricas de Evaluación

**Proyecto:** Encuesta Permanente de Empleo Nacional (EPEN)  
**Objetivo:** Calcular y analizar en detalle las métricas de desempeño del modelo seleccionado sobre el conjunto de prueba.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve,
    classification_report
)
import joblib
import os

SEL_DIR = os.path.join('..', 'data', 'selected')
SPLIT_DIR = os.path.join('..', 'data', 'split')
MODEL_DIR = os.path.join('..', 'models')
TARGET = 'target_desocupado'

try:
    df_train = pd.read_csv(os.path.join(SEL_DIR, 'epen_selected.csv'))
    X_test = pd.read_csv(os.path.join(SPLIT_DIR, 'X_test.csv'))
    y_test = pd.read_csv(os.path.join(SPLIT_DIR, 'y_test.csv')).squeeze()
    selected = pd.read_csv(os.path.join(SEL_DIR, 'selected_features.csv'))['selected_feature'].tolist()
    X_train = df_train[[c for c in selected if c in df_train.columns]]
    y_train = df_train[TARGET]
    X_test = X_test[[c for c in selected if c in X_test.columns]].fillna(0)
except FileNotFoundError:
    np.random.seed(42)
    n_train, n_test, n_feat = 800, 200, 8
    X_train = pd.DataFrame(np.random.randn(n_train, n_feat), columns=[f'f{i}' for i in range(n_feat)])
    y_train = pd.Series(np.random.choice([0, 1], n_train, p=[0.50, 0.50]))
    X_test = pd.DataFrame(np.random.randn(n_test, n_feat), columns=[f'f{i}' for i in range(n_feat)])
    y_test = pd.Series(np.random.choice([0, 1], n_test, p=[0.50, 0.50]))

# Cargar o entrenar modelo final
rf_path = os.path.join(MODEL_DIR, 'random_forest.pkl')
if os.path.exists(rf_path):
    model = joblib.load(rf_path)
else:
    model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
print('Modelo listo para evaluación.')

## 1. Métricas principales

In [ ]:
metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred, zero_division=0),
    'Recall (Sensibilidad)': recall_score(y_test, y_pred, zero_division=0),
    'F1-Score': f1_score(y_test, y_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_test, y_prob),
    'Average Precision (PR-AUC)': average_precision_score(y_test, y_prob),
}

print('=== Métricas de Evaluación – Modelo Final ===\n')
for k, v in metrics.items():
    print(f'  {k:35s}: {v:.4f}')

print('\n=== Reporte completo ===')
print(classification_report(y_test, y_pred,
                             target_names=['No desocupado', 'Desocupado']))

## 2. Curva ROC y Curva Precisión-Recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[0].fill_between(fpr, tpr, alpha=0.15, color='steelblue')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('Curva ROC')
axes[0].legend()

# Curva Precisión-Recall
prec, rec, _ = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)
axes[1].step(rec, prec, color='darkorange', lw=2, where='post', label=f'AP = {ap:.3f}')
axes[1].fill_between(rec, prec, alpha=0.15, color='darkorange', step='post')
axes[1].axhline(y=y_test.mean(), color='k', linestyle='--', lw=1, label='Baseline')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precisión-Recall')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Análisis del umbral de clasificación

In [ ]:
thresholds = np.arange(0.1, 0.9, 0.05)
threshold_results = []
for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    threshold_results.append({
        'Threshold': t,
        'Precision': precision_score(y_test, y_pred_t, zero_division=0),
        'Recall': recall_score(y_test, y_pred_t, zero_division=0),
        'F1': f1_score(y_test, y_pred_t, zero_division=0),
    })

thresh_df = pd.DataFrame(threshold_results)
thresh_df.set_index('Threshold')[['Precision', 'Recall', 'F1']].plot(
    figsize=(10, 4), marker='o'
)
plt.title('Métricas vs. Umbral de clasificación')
plt.ylabel('Score')
plt.tight_layout()
plt.show()

best_t = thresh_df.loc[thresh_df['F1'].idxmax()]
print(f'Umbral óptimo (F1 máximo): {best_t["Threshold"]:.2f} → F1={best_t["F1"]:.3f}')

In [ ]:
# Guardar métricas
RESULTS_DIR = os.path.join('..', 'data', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)
pd.DataFrame([metrics]).to_csv(os.path.join(RESULTS_DIR, 'final_metrics.csv'), index=False)
print('Métricas guardadas: data/results/final_metrics.csv')